In [ ]:
# 统一入口：在这里选择 frame，并查看关键路径信息
from pathlib import Path
import re
import numpy as np

PERSON_ID = '02'
ENV_NAME = '夜多い'
FRAME_IDX = 1500

HEAD3D_ROOT = Path('/workspace/data/head3d_fuse_results')
SAM3D_ROOT = Path('/workspace/data/sam3d_body_results_right_full')

FUSED_DIR = HEAD3D_ROOT / PERSON_ID / ENV_NAME / 'smoothed_fused_npz'
FUSED_FILE_PATH = FUSED_DIR / f'frame_{FRAME_IDX:06d}_fused.npy'

available_frames = []
if FUSED_DIR.exists():
    for p in sorted(FUSED_DIR.glob('frame_*_fused.npy')):
        m = re.search(r'frame_(\d+)_fused\.npy$', p.name)
        if m:
            available_frames.append(int(m.group(1)))

print('=== Frame Selection ===')
print(f'PERSON_ID: {PERSON_ID}')
print(f'ENV_NAME: {ENV_NAME}')
print(f'FRAME_IDX: {FRAME_IDX}')
print(f'FUSED_DIR: {FUSED_DIR} | exists={FUSED_DIR.exists()}')
print(f'FUSED_FILE_PATH: {FUSED_FILE_PATH} | exists={FUSED_FILE_PATH.exists()}')
print(f'available frame count: {len(available_frames)}')
if available_frames:
    print(f'frame range: [{available_frames[0]}, {available_frames[-1]}]')
    print(f'first 10 frames: {available_frames[:10]}')

print('\n=== Expected NPZ Paths ===')
for view in ['front', 'left', 'right']:
    p = SAM3D_ROOT / PERSON_ID / ENV_NAME / view / f'{FRAME_IDX:06d}_sam3d_body.npz'
    print(f'{view:>5}: {p} | exists={p.exists()}')

data = np.load(FUSED_FILE_PATH, allow_pickle=True)

In [ ]:
info = data.item()
info.keys()

In [ ]:
for key in info.keys():
    print(f'{key}: {type(info[key])}')

In [ ]:
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

sys.path.append('/workspace/code')

from vis_3d_kpt.metadata.mhr70_drive import (
    get_head_hand_indices_mapping,
    pose_info as mhr70_pose_info,
 )
from vis_3d_kpt.visualization.scene_visualizer import SceneVisualizer

scene_visualizer = SceneVisualizer(line_width=2, radius=5)
scene_visualizer.set_pose_meta(mhr70_pose_info)


def visualize_part_comparison(
    keypoints_3d: np.ndarray,
    selected_parts=None,
 ):
    mapping = get_head_hand_indices_mapping()

    head_indices = mapping['head']['indices']
    left_hand_indices = mapping['left_hand']['indices']
    right_hand_indices = mapping['right_hand']['indices']
    left_arm_indices = [5, 7, 62]
    right_arm_indices = [6, 8, 41]

    part_style = {
        'head': dict(indices=head_indices, point_color='red', line_color='red', label=f"Head ({len(head_indices)} pts)"),
        'left_hand': dict(indices=left_hand_indices, point_color='blue', line_color='blue', label=f"Left Hand ({len(left_hand_indices)} pts)"),
        'right_hand': dict(indices=right_hand_indices, point_color='green', line_color='green', label=f"Right Hand ({len(right_hand_indices)} pts)"),
        'left_arm': dict(indices=left_arm_indices, point_color='orange', line_color='orange', label='Left Arm (3 pts)'),
        'right_arm': dict(indices=right_arm_indices, point_color='purple', line_color='purple', label='Right Arm (3 pts)'),
    }

    if selected_parts is None:
        selected_parts = ['head', 'left_arm', 'right_arm', 'left_hand', 'right_hand']

    edges, _ = scene_visualizer.get_skeleton_edges_with_colors()

    fig = plt.figure(figsize=(12, 10))
    ax = fig.add_subplot(111, projection='3d')

    for part_name in selected_parts:
        if part_name not in part_style:
            continue
        style = part_style[part_name]
        indices = style['indices']
        part_kpts = keypoints_3d[indices]

        ax.scatter(
            part_kpts[:, 0],
            part_kpts[:, 1],
            part_kpts[:, 2],
            c=style['point_color'],
            s=50,
            alpha=0.95,
            label=style['label'],
        )

        scene_visualizer.draw_skeleton_edges(
            ax=ax,
            keypoints_3d=keypoints_3d,
            edges=edges,
            colors=[style['line_color']] * len(edges),
            part_index_set=set(indices),
            linewidth=2.4,
            alpha=0.9,
        )

    ax.set_xlabel('X')
    ax.set_ylabel('Y')
    ax.set_zlabel('Z')
    # ax.set_title(f"{title_prefix} - Parts in One Coordinate System")
    ax.legend(loc='upper right')
    # ax.view_init(elev=20, azim=45)
    plt.tight_layout()

    return fig, ax

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

# 1) 优先复用第 1 个代码单元的选择
default_file = '/workspace/data/head3d_fuse_results/01/夜多い/smoothed_fused_npz/frame_000623_fused.npy'
FILE_PATH = Path(globals().get('FUSED_FILE_PATH', default_file))

# 2) 加载数据
arr = np.load(FILE_PATH, allow_pickle=True)
sample = arr.item() if isinstance(arr, np.ndarray) and arr.dtype == object else arr

if not isinstance(sample, dict):
    raise TypeError(f'Expected dict in npy, got: {type(sample)}')

if 'fused_keypoints_3d' not in sample:
    raise KeyError("Missing key 'fused_keypoints_3d' in loaded file")

kpts3d = sample['fused_keypoints_3d']
frame_idx = sample.get('frame_idx', 'unknown')

# 3) 可视化（可选: 仅头部 或 头+手臂+手）
selected_parts = ['head', 'left_arm', 'right_arm', 'left_hand', 'right_hand']
# y 和 z 交换显示，并且交换之后的z需要取负数，使得显示更符合常规习惯
kpts3d[:, [1, 2]] = kpts3d[:, [2, 1]]
kpts3d[:, 2] = -kpts3d[:, 2]

fig, ax = visualize_part_comparison(
    keypoints_3d=kpts3d,
    selected_parts=selected_parts,
 )

# 4) 保存图片
save_path = Path('/workspace/code/logs/vis_3d_kpt/part_comparison_single_file.png')
save_path.parent.mkdir(parents=True, exist_ok=True)
plt.savefig(save_path, dpi=150, bbox_inches='tight')
print(f'✅ 已保存: {save_path}')

plt.show()
plt.close(fig)

In [ ]:
from pathlib import Path
import re
import numpy as np
import matplotlib.pyplot as plt

# 优先复用第 1 个代码单元的选择
VIS_FRAME_IDX = int(globals().get('FRAME_IDX', 623))

# 可选: 'head_arm_hand' 或 'all_keep'
VIS_PART_SCHEME = 'head_arm_hand'

# 颜色顺序修正：你当前数据里 frame 是 RGB，不要再做 BGR->RGB 翻转
# 如遇某批数据是 BGR，可改成 'BGR'
FRAME_COLOR_ORDER = 'RGB'

AXIS_PAD_RATIO = 0.05

# 默认从 smoothed_fused_npz 读取，可按需改成 fused_npz
DEFAULT_FUSED_DIR = Path(globals().get('FUSED_DIR', '/workspace/data/head3d_fuse_results/01/夜多い/smoothed_fused_npz'))

def _list_available_frame_indices(fused_dir: Path):
    if not fused_dir.exists():
        raise FileNotFoundError(fused_dir)
    idxs = []
    for p in sorted(fused_dir.glob('frame_*_fused.npy')):
        m = re.search(r'frame_(\d+)_fused\.npy$', p.name)
        if m:
            idxs.append(int(m.group(1)))
    return idxs

def _resolve_file_path(fused_dir: Path, frame_idx: int | None):
    available = _list_available_frame_indices(fused_dir)
    if not available:
        raise RuntimeError(f'目录中没有 frame_*_fused.npy: {fused_dir}')

    if frame_idx is None:
        frame_idx = available[0]

    if frame_idx not in set(available):
        raise ValueError(
            f'frame_idx={frame_idx} 不在可用列表中，示例可用值: {available[:10]} ... 总数={len(available)}'
        )

    file_path = fused_dir / f'frame_{frame_idx:06d}_fused.npy'
    return file_path, frame_idx, available

def _load_fused_sample(file_path: Path):
    arr = np.load(file_path, allow_pickle=True)
    sample = arr.item() if isinstance(arr, np.ndarray) and arr.dtype == object else arr
    if not isinstance(sample, dict):
        raise TypeError(f'Expected dict in npy, got: {type(sample)}')
    return sample

if 'scene_visualizer' not in globals():
    from vis_3d_kpt.metadata.mhr70_drive import pose_info as mhr70_pose_info
    from vis_3d_kpt.visualization.scene_visualizer import SceneVisualizer
    scene_visualizer = SceneVisualizer(line_width=2, radius=5)
    scene_visualizer.set_pose_meta(mhr70_pose_info)

if 'get_head_hand_indices_mapping' not in globals():
    from vis_3d_kpt.metadata.mhr70_drive import get_head_hand_indices_mapping

def _build_active_indices(scheme: str):
    # 与 head3D_fuse 一致的全量保留索引：头部+肩部+双手+肩峰/颈部
    keep_indices_all = list(range(0, 5)) + [5, 6] + list(range(21, 63)) + [67, 68, 69]

    if scheme == 'all_keep':
        indices = keep_indices_all
    elif scheme == 'head_arm_hand':
        mapping = get_head_hand_indices_mapping()
        head = mapping['head']['indices']
        left_hand = mapping['left_hand']['indices']
        right_hand = mapping['right_hand']['indices']
        left_arm = [5, 7, 62]
        right_arm = [6, 8, 41]
        indices = sorted(set(head + left_hand + right_hand + left_arm + right_arm))
    else:
        raise ValueError(f'未知 VIS_PART_SCHEME: {scheme}')

    return indices, set(indices)

def _infer_npz_paths_from_fused_path(fused_path: Path, frame_idx: int):
    person_id = fused_path.parent.parent.parent.name
    env_name = fused_path.parent.parent.name
    sam3d_root = Path('/workspace/data/sam3d_body_results_right_full')
    return {
        view: sam3d_root / person_id / env_name / view / f'{frame_idx:06d}_sam3d_body.npz'
        for view in ['front', 'left', 'right']
    }

FILE_PATH, RESOLVED_FRAME_IDX, AVAILABLE_FRAME_IDXS = _resolve_file_path(DEFAULT_FUSED_DIR, VIS_FRAME_IDX)
sample = _load_fused_sample(FILE_PATH)

npz_paths = sample.get('npz_paths', {})
frame_idx = int(sample.get('frame_idx', RESOLVED_FRAME_IDX))

if not npz_paths:
    npz_paths = _infer_npz_paths_from_fused_path(FILE_PATH, frame_idx)
    print('ℹ️ 当前样本无 npz_paths，已按目录规则自动推断三视角路径')

ACTIVE_KEYPOINT_INDICES, ACTIVE_INDEX_SET = _build_active_indices(VIS_PART_SCHEME)

def _extract_view_data(npz_file_path: Path):
    if not npz_file_path.exists():
        raise FileNotFoundError(npz_file_path)

    frame_img = None
    with np.load(npz_file_path, allow_pickle=True) as z:
        if 'pred_keypoints_3d' in z.files:
            full_k = z['pred_keypoints_3d']
            if 'frame' in z.files:
                frame_img = z['frame']
        elif 'output' in z.files:
            out = z['output'].item()
            if 'pred_keypoints_3d' not in out:
                raise KeyError(f"{npz_file_path} 的 output 中没有 pred_keypoints_3d")
            full_k = out['pred_keypoints_3d']
            frame_img = out.get('frame', None)
        elif 'filtered_pred_keypoints_3d' in z.files:
            full_k = z['filtered_pred_keypoints_3d']
            if 'frame' in z.files:
                frame_img = z['frame']
        else:
            raise KeyError(f"{npz_file_path} 中没有可用 3D 关键点字段")

    if full_k.ndim == 3 and full_k.shape[0] >= 1:
        full_k = full_k[0]
    full_k = np.asarray(full_k)

    if full_k.shape[0] > max(ACTIVE_KEYPOINT_INDICES):
        active_k = full_k[ACTIVE_KEYPOINT_INDICES]
    else:
        raise ValueError(f'关键点数量不足，无法按过滤索引提取: shape={full_k.shape}')

    return full_k, active_k, frame_img

def _compute_global_axis_limits(loaded_samples):
    all_active = np.concatenate([active_k for _, _, _, active_k, _ in loaded_samples], axis=0)
    mins = np.min(all_active, axis=0)
    maxs = np.max(all_active, axis=0)
    spans = np.maximum(maxs - mins, 1e-6)
    pads = spans * AXIS_PAD_RATIO
    return [
        (mins[0] - pads[0], maxs[0] + pads[0]),
        (mins[1] - pads[1], maxs[1] + pads[1]),
        (mins[2] - pads[2], maxs[2] + pads[2]),
    ]

def _show_frame_only(view: str, frame_img: np.ndarray):
    fig = plt.figure(figsize=(6, 5))
    ax = fig.add_subplot(111)
    if frame_img is not None:
        if frame_img.ndim == 3 and frame_img.shape[-1] == 3:
            if FRAME_COLOR_ORDER == 'BGR':
                ax.imshow(frame_img[..., ::-1])
            else:
                ax.imshow(frame_img)
        else:
            ax.imshow(frame_img, cmap='gray')
    else:
        ax.text(0.5, 0.5, 'No frame in npz', ha='center', va='center')
    ax.axis('off')
    plt.tight_layout()
    plt.show()
    plt.close(fig)

def _show_filtered_3d_only(
    view: str,
    full_k: np.ndarray,
    active_k: np.ndarray,
    axis_limits,
):
    # 直接复用上一个 cell 定义的统一绘图函数
    if 'visualize_part_comparison' not in globals():
        raise RuntimeError('请先运行第 4 个代码单元，以定义 visualize_part_comparison')

    # 仅在显示层交换 Y/Z，不改动原始数据
    full_k_plot = full_k[:, [0, 2, 1]].copy()

    # 交换之后的 z 取负，方向与当前视图约定一致
    full_k_plot[:, 2] = -full_k_plot[:, 2]

    selected_parts = ['head', 'left_arm', 'right_arm', 'left_hand', 'right_hand']
    fig, ax = visualize_part_comparison(
        keypoints_3d=full_k_plot,
        selected_parts=selected_parts,
    )

    plt.show()
    plt.close(fig)

view_order = ['front', 'left', 'right']
loaded = []

for view in view_order:
    if view not in npz_paths:
        continue
    p = Path(npz_paths[view])
    full_k, active_k, frame_img = _extract_view_data(p)
    loaded.append((view, p, full_k, active_k, frame_img))

if not loaded:
    raise RuntimeError('没有找到可用的 front/left/right 路径')

global_axis_limits = _compute_global_axis_limits(loaded)

print(f'可用 frame 总数: {len(AVAILABLE_FRAME_IDXS)}')
print(f'当前可视化 frame_idx: {frame_idx} | 文件: {FILE_PATH}')
print(f'部位连线方案: {VIS_PART_SCHEME} | 点数: {len(ACTIVE_KEYPOINT_INDICES)}')
print(f'Frame color order: {FRAME_COLOR_ORDER}')
print(f'统一坐标轴范围: X={global_axis_limits[0]}, Y={global_axis_limits[1]}, Z={global_axis_limits[2]}')
print('输出方式: 每个视角两个独立 figure（frame 和 3D）')
print('\n已加载文件:')
for view, p, full_k, active_k, frame_img in loaded:
    print(f'  - {view:>5}: {p} | full_shape={full_k.shape} | active_shape={active_k.shape} | frame={None if frame_img is None else frame_img.shape}')

for view, p, full_k, active_k, frame_img in loaded:
    _show_frame_only(view, frame_img)
    _show_filtered_3d_only(view, full_k, active_k, axis_limits=global_axis_limits)